## **Embedding Generation Using DINO (Self-Distillation with No Labels)**

In this notebook, we will:
- Load the pre-processed and augmented images.
- Generate embeddings using the DINO model.
- Save the embeddings and related data for evaluation..

---

## **Table of Contents**
---
1. Import Libraries
2. Set Up Device
3. Load Augmented Image Mapping
4. Load DINO Model
5. Generate Embeddings
6. Select Representative Embeddings
7. Save Embeddings
8. Clear Memory
9. Conclusion

---
### **Step 1: Import Libraries**

We begin by importing the necessary libraries.

In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
from torchvision import transforms
import gc

---
### **Step 2: Set Up Device**

In [2]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


---
### **Step 3: Load Augmented Image Mapping**

In [3]:
# Load augmented image mapping
augmented_df = pd.read_csv('augmented_image_mapping.csv')

---
### **Step 4: Load DINO Model**

In [4]:
# Load the DINO model
model = torch.hub.load('facebookresearch/dino:main', 'dino_vitb16').to(device)
model.eval()

# Define the transform
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.228, 0.224, 0.225)),
])

Downloading: "https://github.com/facebookresearch/dino/zipball/main" to /Users/mohammedkhodorfirasal-tal/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dino/dino_vitbase16_pretrain/dino_vitbase16_pretrain.pth" to /Users/mohammedkhodorfirasal-tal/.cache/torch/hub/checkpoints/dino_vitbase16_pretrain.pth
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 327M/327M [00:20<00:00, 16.4MB/s]


---
### **Step 5: Generate Embedding**

In [5]:
# Directory containing augmented images
augmented_dir = './augmented_dataset/'

# Prepare lists to store embeddings and image information
embeddings = []
augmented_image_files = []
original_image_files = []

# Generate embeddings
for idx, row in tqdm(augmented_df.iterrows(), total=len(augmented_df), desc='Generating Embeddings'):
    augmented_image_file = row['augmented_image']
    original_image_file = row['original_image']
    image_path = os.path.join(augmented_dir, augmented_image_file)

    # Load and preprocess image
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)

    # Generate embedding
    with torch.no_grad():
        embedding = model(img_tensor)
        embedding = embedding.cpu().numpy()[0]
        # Normalize embedding
        embedding = embedding / np.linalg.norm(embedding)
        embeddings.append(embedding)
        augmented_image_files.append(augmented_image_file)
        original_image_files.append(original_image_file)

    # Clear memory
    del img, img_tensor, embedding
    torch.cuda.empty_cache()
    gc.collect()

# Convert embeddings to NumPy array
embeddings = np.vstack(embeddings)

Generating Embeddings: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10080/10080 [46:33<00:00,  3.61it/s]


---
### **Step 6: Select Representative Embeddings**

We will use the Highest Norm Criterion to select the most representative embedding for each original image.

In [6]:
# Create a DataFrame for grouping
data = pd.DataFrame({
    'original_image': original_image_files,
    'augmented_image': augmented_image_files,
    'embedding_index': range(len(embeddings))
})

selected_embeddings = []
selected_image_files = []

# Group by original image and select the embedding with the highest norm
for original_image, group in data.groupby('original_image'):
    indices = group['embedding_index'].tolist()
    group_embeddings = embeddings[indices]
    # Compute norms of embeddings
    norms = np.linalg.norm(group_embeddings, axis=1)
    # Select the embedding with the highest norm
    best_idx_in_group = np.argmax(norms)
    best_idx = indices[best_idx_in_group]
    selected_embeddings.append(embeddings[best_idx])
    selected_image_files.append(original_image)

---
### **Step 7: Save Embeddings**

In [7]:
# Convert selected embeddings to NumPy array
selected_embeddings = np.vstack(selected_embeddings)

# Save embeddings and image files
np.save('embeddings_dino.npy', selected_embeddings)
np.save('image_files_dino.npy', selected_image_files)

print('Embeddings for DINO saved.')

Embeddings for DINO saved.


---
### **Step 8: Clear Memory**

In [8]:
# Clear variables and free memory
del embeddings, augmented_image_files, original_image_files, data
del selected_embeddings, selected_image_files, model
torch.cuda.empty_cache()
gc.collect()

0

---
### **Conclusion**

We have generated embeddings using the DINO model, selected representative embeddings, and saved them for evaluation.